# RUKOPYS Qwen3-VL Stage 2B: Build OOF Crop Dataset

This notebook prepares crop-level OOF data for Stage 2B. It reads each `metadata_part*.jsonl` file, materializes the matching crop images in metadata order, and writes one output folder per metadata part. It does not augment, shuffle, rebalance, or replay samples.

Output artifacts per part folder:
- `stage2b_oof_crop_samples.jsonl`
- `metadata_records.jsonl`
- `prompt_config.json`
- `crops/*.jpg`


In [ ]:
# Kaggle dependency cell. GPU is not required; this crop build uses CPU/Pillow.
INSTALL_DEPS = False

if INSTALL_DEPS:
    import subprocess
    import sys

    commands = [
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade-strategy",
            "only-if-needed",
            "pandas==2.2.2",
            "pillow<12",
            "datasets",
        ],
    ]
    for cmd in commands:
        print("Running:", " ".join(cmd), flush=True)
        subprocess.check_call(cmd)


In [ ]:
%pip install pillow

In [ ]:
import hashlib
import json
import re
from pathlib import Path

from PIL import Image

Image.MAX_IMAGE_PIXELS = None

IS_KAGGLE = Path("/kaggle").exists()
WORK_DIR = Path.cwd()

# On Kaggle, leave DATASET_ROOT as /kaggle/input unless you want to point at a specific dataset folder.
DATASET_ROOT = Path("/kaggle/input") if IS_KAGGLE else Path(r"C:\Users\HP\source\rukopys_data")
METADATA_DIR = Path("/kaggle/input") if IS_KAGGLE else WORK_DIR / "OOF"
METADATA_GLOB = "metadata_part*.jsonl"
METADATA_FILES = []  # Optional explicit list, for example: ["/kaggle/input/oof/metadata_part1.jsonl"]

OUTPUT_BASE_DIR = Path("/kaggle/working/oof_stage2b_crops") if IS_KAGGLE else WORK_DIR / "artifacts" / "oof_stage2b_crops"
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

MIN_BOX_SIDE = 8
CROP_PAD_RATIO = 0.035
JPEG_QUALITY = 95

PROMPT_VERSION = "hybrid_prompt_v2"

SOURCE_HINTS = {
    "dictation": "Ukrainian dictation handwriting. Do not complete from canonical text; read only visible characters.",
    "archive": "Historical Ukrainian/Cyrillic document. Preserve old spelling; do not modernize.",
    "school": "School homework. It may contain corrections, teacher marks, formulas, and mixed handwriting/print.",
    "university": "University exam/coursework. It may contain formulas, tables, chemistry notation, and technical symbols.",
}
DEFAULT_SOURCE_HINT = "Read only visible characters from this crop."

SPECIAL_TEXT_MARKER_RULES = (
    "Use [illegible] only for unreadable words inside an otherwise legible text region. "
    "Use ~~word~~ for visible strikethrough and ~~old~~{new} for visible correction."
)

STAGE_B_GUARDRAILS = (
    "The final transcription must be supported by the crop. "
    "Do not complete missing words from source hint, language prior, or canonical dictation text. "
    "Do not translate, correct grammar, normalize spelling, expand abbreviations, summarize, "
    "or infer hidden/missing text. No JSON, no Markdown, no explanation."
)

CROP_PROMPTS = {
    "handwritten": (
        "Transcribe the visible handwritten text exactly. Preserve punctuation, line content, "
        "corrections, spelling mistakes, capitalization, digits, abbreviations, quotes, hyphens, "
        "line-final dashes, visible spacing, and strikethrough markers. Return only text."
    ),
    "printed": (
        "Transcribe the visible printed or typed text exactly. Preserve punctuation, line content, "
        "corrections, spelling mistakes, capitalization, digits, abbreviations, quotes, hyphens, "
        "line-final dashes, visible spacing, and strikethrough markers. Return only text."
    ),
    "annotation": "Read this short annotation or teacher mark. Return only the exact visible text.",
    "formula": (
        "Read this standalone math, logic, vector, matrix, determinant, set/relation, statistics, physics, "
        "or chemistry expression exactly as written. Return only formula text, using LaTeX when it is the "
        "clearest representation and plain Unicode when it better matches the handwriting. Preserve visible "
        "symbols, indices, superscripts, subscripts, arrows, fractions, matrix/determinant structure, punctuation, "
        "numbering, and strikethrough/correction markers. Do not solve, simplify, normalize, explain, or convert "
        "old notation into a different style."
    ),
    "table": (
        "Read this table region exactly. Return only pipe-separated table text. Use one output line per visual row "
        "and | between cells. Preserve empty cells with empty fields, for example A||C. Preserve row order, "
        "column order, multi-word cell text, wrapped cell text, numbers, units, punctuation, dashes, and visible "
        "spelling mistakes. Do not infer missing cells, do not rebalance columns, do not summarize, and do not explain."
    ),
    "image": "Return an empty string.",
    "graph": "Return an empty string.",
    "default": "Transcribe the visible content exactly. Preserve punctuation, corrections, and visible spacing. Return only text.",
}

VALID_TYPES = {"handwritten", "printed", "formula", "table", "annotation", "image", "graph"}
SCORABLE_TYPES = {"handwritten", "printed", "formula", "table", "annotation"}
IMAGE_EXTENSIONS = [".png", ".jpg", ".jpeg", ".webp", ".bmp"]


In [ ]:

def unique_existing_paths(paths):
    seen = set()
    out = []
    for path in paths:
        p = Path(path)
        key = str(p.resolve()) if p.exists() else str(p)
        if key not in seen and p.exists():
            out.append(p)
            seen.add(key)
    return out


def discover_metadata_files():
    if METADATA_FILES:
        files = [Path(p) for p in METADATA_FILES]
    else:
        metadata_dir = Path(METADATA_DIR)
        files = sorted(metadata_dir.rglob(METADATA_GLOB)) if metadata_dir.exists() else []
    files = [p for p in files if p.exists()]
    if not files:
        raise FileNotFoundError(f"No files matched {METADATA_GLOB!r} under {METADATA_DIR}")
    return files


def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def iter_image_candidates(root, file_name):
    raw = Path(file_name)
    name = raw.name
    stem = raw.stem
    candidate_names = [name] + [stem + ext for ext in IMAGE_EXTENSIONS if stem + ext != name]
    rel_paths = []
    for candidate_name in candidate_names:
        rel_paths.append(raw.parent / candidate_name if str(raw.parent) != "." else Path(candidate_name))
    for split in ["train", "silver", "sliver", "test", "validation"]:
        for candidate_name in candidate_names:
            rel_paths.extend([Path(split) / "images" / candidate_name, Path(split) / candidate_name])
    for candidate_name in candidate_names:
        rel_paths.extend([Path("images") / candidate_name, Path(candidate_name)])

    seen = set()
    for rel_path in rel_paths:
        candidate = Path(root) / rel_path
        key = str(candidate)
        if key not in seen:
            yield candidate
            seen.add(key)


def resolve_image_path(root, file_name):
    for path in iter_image_candidates(root, file_name):
        if path.exists():
            return str(path)
    return str(next(iter_image_candidates(root, file_name)))


def first_file_names(metadata_files, limit=12):
    names = []
    for metadata_path in metadata_files:
        with open(metadata_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                row = json.loads(line)
                if row.get("file_name"):
                    names.append(row["file_name"])
                    break
        if len(names) >= limit:
            break
    return names


def infer_dataset_root(metadata_files):
    candidates = [Path(DATASET_ROOT)]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend(sorted([p for p in kaggle_input.iterdir() if p.is_dir()]))
        candidates.append(kaggle_input)
    candidates.extend([WORK_DIR, WORK_DIR / "dataset", WORK_DIR.parent])
    candidates = unique_existing_paths(candidates)

    probe_names = first_file_names(metadata_files)
    for root in candidates:
        if any(any(path.exists() for path in iter_image_candidates(root, name)) for name in probe_names):
            print("Using dataset root:", root)
            return root
    fallback = Path(DATASET_ROOT)
    print("Could not verify image root from metadata; using configured DATASET_ROOT:", fallback)
    return fallback


def clamp_box(box, w, h):
    if not isinstance(box, list) or len(box) != 4:
        return None
    try:
        x1, y1, x2, y2 = [float(v) for v in box]
    except Exception:
        return None
    x1, x2 = sorted((max(0, min(w, x1)), max(0, min(w, x2))))
    y1, y2 = sorted((max(0, min(h, y1)), max(0, min(h, y2))))
    if x2 - x1 < MIN_BOX_SIDE or y2 - y1 < MIN_BOX_SIDE:
        return None
    return [int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))]


def normalize_type(value):
    value = str(value or "handwritten").strip().lower()
    return value if value in VALID_TYPES else "handwritten"


def normalize_source(value):
    value = str(value or "").strip().lower()
    return value if value in SOURCE_HINTS else "default"


def get_source_hint(source):
    return SOURCE_HINTS.get(normalize_source(source), DEFAULT_SOURCE_HINT)


def build_crop_prompt(rtype, source=None):
    rtype = normalize_type(rtype)
    type_prompt = CROP_PROMPTS.get(rtype, CROP_PROMPTS["default"])
    if rtype in {"image", "graph"}:
        return type_prompt
    return "\n".join([get_source_hint(source), type_prompt, SPECIAL_TEXT_MARKER_RULES, STAGE_B_GUARDRAILS])


def text_is_sane(text):
    if text is None:
        return False
    text = str(text).strip()
    if not text or len(text) > 260:
        return False
    if re.search(r"(.)\1{10,}", text):
        return False
    bad = sum(ch in "\ufffdï¿½" for ch in text)
    return bad == 0


def region_is_crop_candidate(region):
    rtype = normalize_type(region.get("type"))
    if rtype not in SCORABLE_TYPES:
        return False
    if str(region.get("language", "uk")).lower() == "other":
        return False
    if str(region.get("legibility", "legible")).lower() == "illegible":
        return False
    return text_is_sane(region.get("text"))



In [ ]:

def make_crop_sample(record, region, root):
    w = max(1, int(record.get("image_width") or 1))
    h = max(1, int(record.get("image_height") or 1))
    box = clamp_box(region.get("bbox"), w, h)
    if box is None or not region_is_crop_candidate(region):
        return None
    rtype = normalize_type(region.get("type"))
    return {
        "task": "crop_ocr",
        "source_image_path": resolve_image_path(root, record["file_name"]),
        "source_file_name": record["file_name"],
        "bbox": box,
        "region_type": rtype,
        "prompt": build_crop_prompt(rtype, source=record.get("source")),
        "answer": str(region.get("text") or ""),
        "source": record.get("source", "unknown"),
    }


def build_samples(records, root, part_name):
    crop_samples = []
    for record in records:
        for region in record.get("regions") or []:
            crop = make_crop_sample(record, region, root)
            if crop is not None:
                crop_samples.append(crop)
    print(f"{part_name}: crop={len(crop_samples)}")
    return crop_samples


In [ ]:

def crop_with_padding(sample, pad_ratio=CROP_PAD_RATIO):
    with Image.open(sample["source_image_path"]) as img:
        img = img.convert("RGB")
        w, h = img.size
        x1, y1, x2, y2 = sample["bbox"]
        pad = int(round(max(x2 - x1, y2 - y1) * pad_ratio))
        x1 = max(0, x1 - pad)
        y1 = max(0, y1 - pad)
        x2 = min(w, x2 + pad)
        y2 = min(h, y2 + pad)
        crop = img.crop((x1, y1, x2, y2))
    return crop


def sample_cache_name(sample, idx):
    key = json.dumps({
        "file": sample.get("source_file_name"),
        "bbox": sample.get("bbox"),
        "type": sample.get("region_type"),
    }, sort_keys=True, ensure_ascii=False)
    digest = hashlib.sha1(key.encode("utf-8")).hexdigest()[:12]
    rtype = re.sub(r"[^a-z0-9_-]+", "_", sample.get("region_type", "crop"))
    return f"{idx:06d}_{rtype}_{digest}.jpg"


def materialize_crops(samples, output_dir):
    crop_dir = output_dir / "crops"
    crop_dir.mkdir(parents=True, exist_ok=True)
    manifest = []
    for idx, sample in enumerate(samples):
        rel_path = Path("crops") / sample_cache_name(sample, idx)
        out_path = output_dir / rel_path
        if not out_path.exists():
            crop = crop_with_padding(sample)
            crop.save(out_path, quality=JPEG_QUALITY, optimize=True)
        row = dict(sample)
        row["image_path"] = str(rel_path).replace("\\", "/")
        row["relative_image_path"] = row["image_path"]
        row.pop("source_image_path", None)
        manifest.append(row)
        if (idx + 1) % 1000 == 0:
            print(f"materialized {idx + 1}/{len(samples)} crops", flush=True)
    return manifest


In [ ]:
def output_dir_for_metadata(metadata_path):
    stem = Path(metadata_path).stem
    match = re.fullmatch(r"metadata_(part\d+)", stem)
    folder_name = f"oof_{match.group(1)}" if match else f"oof_{re.sub(r'[^a-zA-Z0-9_-]+', '_', stem)}"
    return OUTPUT_BASE_DIR / folder_name


def assert_source_images_exist(samples, part_name, limit=8):
    missing = [s["source_image_path"] for s in samples if not Path(s["source_image_path"]).exists()]
    if missing:
        preview = "\n".join(missing[:limit])
        raise FileNotFoundError(f"{part_name}: missing {len(missing)} source images. First missing paths:\n{preview}")


def write_prompt_config(path, metadata_path, records, manifest, root):
    path.write_text(
        json.dumps({
            "prompt_version": PROMPT_VERSION,
            "stage": "stage2b_oof_crop_dataset",
            "metadata_file": str(metadata_path),
            "dataset_root": str(root),
            "source_hints": SOURCE_HINTS,
            "default_source_hint": DEFAULT_SOURCE_HINT,
            "crop_prompts": CROP_PROMPTS,
            "special_text_marker_rules": SPECIAL_TEXT_MARKER_RULES,
            "stage_b_guardrails": STAGE_B_GUARDRAILS,
            "crop_prompt_assembly": "source_hint + type_prompt + special_text_marker_rules + stage_b_guardrails",
            "crop_pad_ratio": CROP_PAD_RATIO,
            "page_count": len(records),
            "sample_count": len(manifest),
        }, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def build_metadata_part(metadata_path, root):
    metadata_path = Path(metadata_path)
    part_name = metadata_path.stem
    output_dir = output_dir_for_metadata(metadata_path)
    output_dir.mkdir(parents=True, exist_ok=True)

    records = read_jsonl(metadata_path)
    print(f"\nBuilding {part_name}: pages={len(records)} -> {output_dir}")
    samples = build_samples(records, root, part_name)
    if not samples:
        raise RuntimeError(f"{part_name}: no Stage 2B crop samples were built. Check metadata and filters.")
    assert_source_images_exist(samples, part_name)

    manifest = materialize_crops(samples, output_dir)
    samples_path = output_dir / "stage2b_oof_crop_samples.jsonl"
    records_path = output_dir / "metadata_records.jsonl"
    prompt_config_path = output_dir / "prompt_config.json"

    write_jsonl(samples_path, manifest)
    write_jsonl(records_path, records)
    write_prompt_config(prompt_config_path, metadata_path, records, manifest, root)

    print("Wrote samples:", samples_path, "rows=", len(manifest))
    print("Wrote metadata records:", records_path, "rows=", len(records))
    print("Wrote prompt config:", prompt_config_path)
    print("Crop dir:", output_dir / "crops")
    return {
        "metadata_path": metadata_path,
        "output_dir": output_dir,
        "manifest": manifest,
        "samples_path": samples_path,
        "prompt_config_path": prompt_config_path,
    }


metadata_files = discover_metadata_files()
print("Metadata files:", [str(p) for p in metadata_files])
root = infer_dataset_root(metadata_files)
run_outputs = [build_metadata_part(path, root) for path in metadata_files]
print("\nFinished folders:", [str(item["output_dir"]) for item in run_outputs])


In [ ]:
# Optional smoke check: open the first cached crop from the first output folder and print its training prompt.
RUN_SMOKE_CHECK = True

if RUN_SMOKE_CHECK and run_outputs and run_outputs[0]["manifest"]:
    first_output = run_outputs[0]
    first = first_output["manifest"][0]
    print(json.dumps({k: first[k] for k in ["image_path", "region_type", "source", "answer"]}, ensure_ascii=False, indent=2))
    print("Prompt preview:\n", first["prompt"][:1200])
    display(Image.open(first_output["output_dir"] / first["image_path"]))
